# D2-01 Day 1 recap and readiness checks

Run this notebook in the `bw` environment. The goal is not to learn anything new yet, but to make sure the Day 1 foundation is still in place before we start using `edges`.


## Learning goals

After this notebook, you should be able to:

- recall the Day 1 workflow from project selection to LCA score
- connect `brightway` objects back to the matrix formulation from Day 1


## Background references

- Mutel, C. (2017). *Brightway: An open source framework for Life Cycle Assessment*. Journal of Open Source Software, 2(12), 236. https://doi.org/10.21105/joss.00236
- Heijungs, R., & Suh, S. (2002). *The Computational Structure of Life Cycle Assessment*. Kluwer Academic Publishers. https://doi.org/10.1007/978-94-015-9900-9
- Sacchi, R., Menacho, A. H., Seitfudem, G., Agez, M., Schlesinger-Martinat, J., Koyamparambath, A., Saldivar, J. S., Loubet, P., & Bauer, C. (2025). Contextual LCIA without the overhead: an exchange-based framework for flexible impact assessment. *The International Journal of Life Cycle Assessment, 30*(12), 3087-3101. https://doi.org/10.1007/s11367-025-02551-7


In [ ]:
import importlib
import importlib.metadata as md
import pandas as pd

packages = ['bw2data', 'bw2calc', 'bw2io', 'bw2analyzer', 'edges']
rows = []
for package in packages:
    spec = importlib.util.find_spec(package)
    rows.append({
        'package': package,
        'available': spec is not None,
        'version': md.version(package) if spec is not None else None,
    })

pd.DataFrame(rows)


## 1) Check the current project and the available databases

Day 2 assumes that Day 1 left you with a usable project, a biosphere database, and a BAFU foreground or background database. We also want at least one LCIA method available.


In [1]:
import importlib
import importlib.metadata as md
import pandas as pd

import bw2data as bd

bd.projects.set_current("aalborg-rlcia-2026")

print('Current project:', bd.projects.current)
print('Number of databases:', len(bd.databases))
print('Number of methods:', len(bd.methods))

database_names = sorted(bd.databases)
print('\nDatabases in the current project:')
for name in database_names:
    print('-', name)

bafu_candidates = [name for name in database_names if name.startswith('bafu')]
print('\nBAFU database candidates:', bafu_candidates)


/opt/homebrew/Caskroom/miniforge/base/envs/bw/lib/python3.11/site-packages/scikits/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__('pkg_resources').declare_namespace(__name__)


Current project: aalborg-rlcia-2026
Number of databases: 3
Number of methods: 669

Databases in the current project:
- bafu
- carbon fiber
- ecoinvent-3.10-biosphere

BAFU database candidates: ['bafu']


## 2) Recall the Day 1 workflow

1. choose a project
2. select a database and an activity
3. define a demand
4. choose a method
5. run `lci()` and `lcia()`
6. inspect the score, matrices, and contributions


## Checkpoint 1

In your own words, answer the following three prompts:

- What is an **activity** in `brightway`?
- What is an **exchange**?
- What is built during `lci()`?
- What changes when we move from `lci()` to `lcia()`?


In [ ]:
recap_answers = {
    'activity': 'An activity is a process-like dataset that produces a reference product and stores metadata such as name, unit, and location.',
    'exchange': 'An exchange is one row linking an activity to an input or an output, such as a technosphere input, a biosphere flow, or a production exchange.',
    'lci': 'lci() builds the supply and inventory results from the demand and the database structure.',
    'lci_to_lcia': 'lcia() applies characterization factors to the inventory and turns it into impact scores.',
}
recap_answers


## 3) Re-run one simple LCA as a recap

We now repeat a very small Day 1 workflow with a gasoline passenger-car activity from the BAFU database. The goal is not the number itself, but to verify that you can remember how to:

- search for an activity
- select a method
- create a demand dictionary
- compute a score
- inspect key matrix shapes


In [2]:
import bw2calc as bc

db = bd.Database("bafu")

activity = next(
    act for act in db
    if 'passenger car' in act['name'].lower()
    and 'operation' in act['name'].lower()
    and 'petrol' in act['name'].lower()
)
method = [
    m for m in bd.methods
    if "GWP" in str(m)
][0]

lca = bc.LCA({activity: 1}, method)
lca.lci()
lca.lcia()

print('Activity:', activity['name'])
print('Method:', method)
print('Score:', lca.score)
print('Technosphere shape:', lca.technosphere_matrix.shape)
print('Biosphere shape:', lca.biosphere_matrix.shape)
print('Characterization matrix shape:', lca.characterization_matrix.shape)
print('Characterized inventory shape:', lca.characterized_inventory.shape)


Activity: Operation, passenger car, petrol, 15% vol. ETBE with ethanol from biomass, EURO4
Method: ('CML v4.8 2016 no LT', 'climate change no LT', 'global warming potential (GWP100) no LT')
Score: 0.2400473341087453
Technosphere shape: (11747, 11747)
Biosphere shape: (1807, 11747)
Characterization matrix shape: (1807, 1807)
Characterized inventory shape: (1807, 11747)


## 5) Reconnect this workflow to the matrix notation

The object mapping from Day 1 is still the same on Day 2:

- the demand dictionary defines `f`
- the technosphere matrix is `A`
- the biosphere matrix is `B`
- `lci()` solves for the scaling vector `s`
- `lcia()` adds characterization factors and gives the impact score

Day 2 will change the characterization step, not the existence of these building blocks.


## Common mistakes

- You can compute an inventory, but no LCIA methods are available in the current project.
- The activity search returns an empty list because the search string is too narrow.


## Recap

After this notebook, you should now be able to:

- confirm that Day 1 packages and data objects are available
- restate the project -> database -> activity -> demand -> method -> score workflow
- re-run a simple `brightway` calculation from memory
- explain that Day 2 extends the characterization step rather than replacing the whole LCA workflow
